In [ ]:
%autoreload
class GBoost(BaseEstimator, RegressorMixin):
    def __init__(self, classifier = 'stump', n_estimators = 10, learning_rate = 0.5, max_depth = 1, 
                 subsample = 1, epsilon = 0.5, beta = 0.5, min_samples_leaf = 0, min_samples_split = 0, 
                 loss_function = 'MSE', validation_fraction = None, patience = None, tolerance = None,
                 hidden_layer_sizes=(10,), activation='relu', optimiser = None,
                 beta2 =0.9, beta3 = 0.999, epsilon2 = 1e-8, eta2=0.01, n_epochs=1000, batch_size=None,
                 seed = 21, measure = 'MSE',
                 decay_function = None, beta_dist = None, decay = None, decay_steps = 1, power = 1, epsilon3 = 1e-8):
        
        self.classifier = classifier
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.epsilon = epsilon #for huber loss
        self.beta = beta #for huber loss
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.subsample = subsample
        self.loss_function = loss_function
        self.seed = seed
        self.measure = measure
        self.stumps = []
        self.loss_history = []  #records loss for plotting
        self.initial_pred = None
        self.current_pred = None
        self.root = None
        self.trees = []
        self.validation_fraction = validation_fraction
        self.validation_scores = []
        self.best_val_score = float('inf')
        self.patience = patience #(no iteration change)
        self.tolerance = tolerance
        self.best_iteration = 0
        self.no_improvement_count = 0
        #learning schedule params
        self.beta_dist = beta_dist
        self.distribution = None
        self.eta_history = []
        self.eta_testing_history = []
        self.decay = decay
        self.decay_steps = decay_steps
        self.power = power
        self.decay_function = decay_function
        self.estimator_preds = [] #training preds based on current estimator
        self.testing_preds = [] #round by round predictions on test data for post hoc analysis
        self.initial_learning_rate = self.learning_rate #for exponential decay
        self.eta = [] #store list of size y, with the learning rate. For adjusting learning rate dynamically on each sample
        self.G = [] #for adaGrad
        self.epsilon3 = epsilon3 #modified adative gradient
        self.test_eta = []
        #MLP params
        self.hidden_layer_sizes = hidden_layer_sizes
        self.optimiser = optimiser
        self.activation = activation
        self.eta2 = eta2
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.weights = []
        self.biases = []
        self.delta_history = []
        self.weights_history = []
        self.biases_history = []
        #MLP adam
        self.beta2 = beta2
        self.beta3 = beta3
        self.epsilon2 = epsilon2
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = [] 
        
        # Set random seed if provided
        if self.seed:
            np.random.seed(self.seed)
        
    def _compute_residuals(self, y, y_pred): 
#         print(f'_compute_residuals: len y: {len(y)}, len y_pred {len(y_pred)}')
        if self.loss_function is 'huber': return self._huber_gradient((y - y_pred))
        else: return y - y_pred
        
    def _get_subsample(self, y): #return indices of sub sampled data
        np.random.seed(self.seed)
        return np.array(np.random.choice(len(y), size=(round(len(y) * self.subsample)), replace=False))
    
    def _initialize_predictions(self, y): #also use for leaf value
        if self.loss_function is 'MAE': return np.median(y)
        elif self.loss_function is 'huber': return self._huber_initialise_predictions(y)
        else: return y.mean()   
    
    def _huber_initialise_predictions(self, y): #to start the preds
        best_loss = float('inf')
        ip = None
        for i in y:
            y_t = self._huber_loss(i) #loss at this point
            total_loss = self._huber_loss(y - y_t) #loss across all data using that point
            if total_loss < best_loss: #minimise loss over data
                best_loss = total_loss
                ip = i                 #get minimising initial prediction         
        return ip
    
    #=======================================LOSS FUNCTIONS=======================================
    def _huber_gradient(self, residuals): #transforms residuals inot a huber gradient. 
        abs_resid = np.abs(residuals)
        gradient = np.where(abs_resid <= self.epsilon, -residuals, -self.epsilon * np.sign(residuals))
        return gradient
    
    def _huber_loss(self, residuals): #to find leaf values with huber loss minimisation
        abs_resid = np.abs(residuals)
        quadratic = self.beta * residuals**2
        linear = self.epsilon * (abs_resid - 0.5 * self.epsilon)
        return np.where(abs_resid <= self.epsilon, quadratic, linear).sum()
    
    def _mse_(self, residuals, left_mask, right_mask):
        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        mse = (np.mean((residuals[left_mask] - left_value) ** 2) * np.sum(left_mask) + 
           np.mean((residuals[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)
        return mse, left_value, right_value

    def _abs_error(self, residuals, left_mask, right_mask): #note, residuals have to be median, not mean for abs_error: outliers
        sign_vector = np.sign(residuals)

        left_value = sign_vector[left_mask].mean()
        right_value = sign_vector[right_mask].mean()

        mae = (np.mean((sign_vector[left_mask] - left_value) ** 2) * np.sum(left_mask) +
               np.mean((sign_vector[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)

        return mae, left_value, right_value

    def _friedman_MSE(self, residuals, left_mask, right_mask, weights = None):
        if weights is None:
            weights = np.ones(len(residuals))

        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        w_l = np.sum(weights[left_mask]) 
        w_r = np.sum(weights[right_mask])

        mean_l = np.mean(residuals[left_mask])
        mean_r = np.mean(residuals[right_mask])

        friedman_mse = ((w_l * w_r) / (w_l + w_r)) * ((mean_l - mean_r)**2)

        return friedman_mse, left_value, right_value

    def _huber(self, residuals, left_mask, right_mask):
        huber_residuals = self._huber_gradient(residuals) #always huber-ise the residuals, even when already hubered from r-1 

        left_value = np.mean(huber_residuals[left_mask]) if np.any(left_mask) else 0
        right_value = np.mean(huber_residuals[right_mask]) if np.any(right_mask) else 0
        #need to figure out best huber loss over all predictions. 
        y_pred = np.full_like(huber_residuals, left_value)
        y_pred[right_mask] = right_value
        
        error = self._huber_loss(self._compute_residuals(huber_residuals, y_pred))
        
        return error, left_value, right_value   
    
    #=======================================STUMP FUNCTIONS=======================================
    def _train_weak_learner(self, X, residuals):
        X = np.array(X)  # Ensure X is a NumPy array
        n_samples, n_features = X.shape
        best_feature_index = None
        best_threshold = None
        best_left_value = None
        best_right_value = None

        if self.loss_function is 'friedmanMSE': best_error = float('-inf')
        else: best_error = float('inf')

        # Iterate over each feature
        for feature_index in range(n_features):
            thresholds = np.unique(X[:, feature_index])

            # Test each threshold as a potential split point
            for threshold in thresholds:
                left_mask = X[:, feature_index] <= threshold
                right_mask = X[:, feature_index] > threshold

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                    
                #skip if either side has fewer samples than the specified minimum samples per leaf
                if (np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf): continue
                    
                #impurity calculation
                if self.loss_function is 'MSE': error, left_value, right_value =  self._mse_(residuals, left_mask, right_mask)
                elif self.loss_function is 'friedmanMSE': error, left_value, right_value =  self._friedman_MSE(residuals, left_mask, right_mask)
                elif self.loss_function is 'MAE': error, left_value, right_value =  self._abs_error(residuals, left_mask, right_mask)
                elif self.loss_function is 'huber': error, left_value, right_value = self._huber(residuals, left_mask, right_mask)

                    
    #             If this is the best split so far, store it. FreidmanMSE maximises, so we need a different approach
                if self.loss_function is 'friedmanMSE':
                    if error > best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value
                else:
                    if error < best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value

        return best_feature_index, best_threshold, best_left_value, best_right_value, best_error
    
    def _fit_stump(self, t, X, residuals):
        if self.subsample is not None:
            indices = self._get_subsample(residuals) #get indices
            resid_subsample = residuals[indices]

            stump = self._train_weak_learner(X[indices], resid_subsample)
            feature_index, threshold, left_value, right_value, best_error = stump

            current_pred_subsample = self.current_pred[indices]
            updates = self._update_predictions(current_pred_subsample, feature_index, threshold, left_value, right_value, X[indices])
            self.current_pred[indices] = updates
            #========================
        else: 
            stump = self._train_weak_learner(X, residuals)
            feature_index, threshold, left_value, right_value, best_error = stump
            self.current_pred = self._update_predictions(self.current_pred, feature_index, threshold, left_value, right_value, X)
        #========================
        self.stumps.append(stump)
        self.estimator_preds.append(self.current_pred)
        self.loss_history.append(best_error)
        return self
    
    #=======================================TREE FUNCTIONS=======================================
    def _leaf_value(self, y): return np.mean(y) #y is residuals for GBoost, can use initialise because the prediction is the mean
    
    def _grow_tree(self, X, residuals, depth = 0): #recursively grow the decision tree up to the specified depth. Which is a stump if not specified
#       #we will recursively get subsamples of the branches to stop braodcasting error. If subsample =1, we use all data
        indices = self._get_subsample(residuals) #get indices
        residuals = residuals[indices]
        X = X[indices]
        n_samples, n_features = X.shape
        
        #check stopping criteria
        if (depth >= self.max_depth or n_samples < self.min_samples_split): 
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        #find best split
        best_feature, best_threshold, left_value, right_value, best_error = self._train_weak_learner(X, residuals)
        
        #if no split found, create a leaf
        if best_feature is None: 
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        y_pred =  self.current_pred[indices]
        y_pred = self._update_predictions(y_pred, best_feature, best_threshold, left_value, right_value, X)
        self.current_pred[indices] = y_pred
#       self.current_pred = self._update_predictions(self.current_pred, best_feature, best_threshold, left_value, right_value, X)
        
        #split data
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = X[:, best_feature] > best_threshold
        left_subtree = self._grow_tree(X[left_mask], residuals[left_mask], depth +1)
        right_subtree = self._grow_tree(X[right_mask], residuals[right_mask], depth +1)
                
        #track the tree stats
        return {
            'feature': best_feature,
            'threshold': best_threshold,
            'left value': left_value,
            'right value': right_value,
            'left': left_subtree,
            'right': right_subtree,
            'error': best_error,
            'is_leaf': False,
            'predictions':self.current_pred
        }
    
    def _fit_tree(self, t, X, residuals):
        self.root = self._grow_tree(X, residuals)
        tree = {}
        tree[f'tree {t}'] = self.root
        self.loss_history.append(self.root['error'])
        self.trees.append(tree)
        self.estimator_preds.append(self.current_pred)
        return self
            
    #=====================================+==MLP FUNCTIONS=======================================
    def _initialize_parameters(self, n_features):
        """Initialize weights and biases for each layer using He initialization."""
        layer_sizes = [n_features] + list(self.hidden_layer_sizes) +[1]  # Output layer has 1 neuron for regression
        self.weights = []
        self.biases = []
        
        #adam
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        for i in range(len(layer_sizes) - 1):
            fan_in = layer_sizes[i]
            fan_out = layer_sizes[i+1]
            # He initialization for ReLU
            if self.activation == 'relu':
                limit = np.sqrt(2.0 / fan_in)
                W = np.random.normal(0, limit, (fan_in, fan_out))
            else:
                # Xavier/Glorot for other activations
                limit = np.sqrt(6.0 / (fan_in + fan_out))
                W = np.random.uniform(-limit, limit, (fan_in, fan_out))
            b = np.zeros((1, fan_out))
            self.weights.append(W)
            self.biases.append(b)
            
            #adam
            self.m_weights.append(np.zeros((fan_in, fan_out)))
            self.v_weights.append(np.zeros((fan_in, fan_out)))
            self.m_biases.append(np.zeros((1, fan_out)))
            self.v_biases.append(np.zeros((1, fan_out)))
    
    def _forward_pass(self, X):
        """Perform forward propagation through the network."""
        activations = [X]
        zs = []  # Linear combinations before activation
        
        # Forward pass through hidden layers
        for i in range(len(self.weights) - 1):
            z = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            zs.append(z)
            # Apply activation function
            if self.activation == 'relu':
                a = np.maximum(0, z)
            elif self.activation == 'tanh':
                a = np.tanh(z)
            elif self.activation == 'sigmoid':
                a = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            else:
                raise ValueError(f"Unsupported activation function: {self.activation}")
            activations.append(a)
        
        # Output layer (no activation for regression)
        z_output = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        zs.append(z_output)
        y_pred = z_output  # Linear output for regression
        activations.append(y_pred)
        
        return activations, zs
    
    def _compute_loss(self, y, residuals):
        if self.loss_function is 'MSE': return np.mean((y - residuals) ** 2)
        elif self.loss_function is 'MAE': return np.mean(abs(y - residuals))
        elif self.loss_function is 'huber': return _huber_loss(residuals)        
    
    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def _get_delta_update(self, delta, weights, z):
        if self.activation == 'relu': return np.dot(delta, weights.T) * self._relu_derivative(z)
        elif self.activation == 'tanh': return np.dot(delta, weights.T) * (z > 0).astype(float)
        elif self.activation == 'sigmoid':
            s = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            return np.dot(delta, weights.T) * s * (1-s)
            
    
    def _backward_pass(self, X, y, activations, zs, time=None):
        """Perform backpropagation to compute gradients."""
        m = X.shape[0]  # Number of samples
        deltas = []
        
        if self.optimiser is None:
            # Output layer delta (for regression, derivative of MSE w.r.t. z)
            delta = (activations[-1] - y) / m
            deltas.append(delta)

            # Backpropagate through hidden layers
            for l in range(len(self.weights) - 2, -1, -1):
                delta = self._get_delta_update(delta, self.weights[l+1], zs[l]) #get delta activation for current layer
                deltas.append(delta)

            # Reverse deltas to match layer order
            deltas.reverse()
            self.delta_history.append(deltas)
            
            # Compute gradients for weights and biases
            for l in range(len(self.weights)):
                # Gradient of loss w.r.t. weights
                dW = np.dot(activations[l].T, deltas[l]) / m
                # Gradient of loss w.r.t. biases
                db = np.sum(deltas[l], axis=0, keepdims=True) / m
                # Update weights and biases
                self.weights[l] -= self.eta2 * dW 
                self.biases[l] -= self.eta2 * db
            
        elif self.optimiser is 'SGD':
            y_pred = activations[-1]

            delta = y_pred - y
            dW = np.dot(activations[-2].T, delta) / X.shape[0]
            db = np.sum(delta, axis=0) / X.shape[0]
            self.weights[-1] -= self.eta2 * dW
            self.biases[-1] -= self.eta2 * db

            for l in range(len(self.weights) - 2, -1, -1):
                delta = self._get_delta_update(delta, self.weights[l+1], zs[l]) #get delta activation for current layer
                deltas.append(delta)
                self.delta_history.append(deltas)
                dW = np.dot(activations[l].T, delta) / X.shape[0]
                db = np.sum(delta, axis=0) / X.shape[0]

                self.weights[l] -= self.eta2 * dW
                self.biases[l] -= self.eta2 * db
                
            self.weights_history.append([w.copy() for w in self.weights])
            self.biases_history.append([b.copy() for b in self.biases])

        elif self.optimiser is 'adam':
            y_pred = activations[-1] # Output of the network

            # 2. backpropagation (using Adam slightly different)
            delta = y_pred - y
            dW = np.dot(activations[-2].T, delta) / X.shape[0] # Average over batch
            db = np.sum(delta, axis=0) / X.shape[0]

            # apply Adam updates to weights
            self.m_weights[-1] = self.beta2 * self.m_weights[-1] + (1 - self.beta2) * dW
            self.v_weights[-1] = self.beta3 * self.v_weights[-1] + (1 - self.beta3) * (dW ** 2)
            m_w_hat = self.m_weights[-1] / (1 - self.beta2**time)
            v_w_hat = self.v_weights[-1] / (1 - self.beta3**time)
            self.weights[-1] -= (self.eta2 * m_w_hat ) / (np.sqrt(v_w_hat) + self.epsilon2)

            # apply Adam updates to bias
            self.m_biases[-1] = self.beta2 * self.m_biases[-1] + (1 - self.beta2) * db
            self.v_biases[-1] = self.beta3 * self.v_biases[-1] + (1 - self.beta3) * (db ** 2)
            m_b_hat = self.m_biases[-1] / (1 - self.beta2**time)
            v_b_hat = self.v_biases[-1] / (1 - self.beta3**time)
            self.biases[-1] -= (self.eta2 * m_b_hat) / (np.sqrt(v_b_hat) + self.epsilon2)

            # Propagate gradients backward through hidden layers
            for l in range(len(self.weights) - 2, -1, -1):
                delta = self._get_delta_update(delta, self.weights[l+1], zs[l]) #get delta activation for current layer
                deltas.append(delta)
                self.delta_history.append(deltas)
                dW_hidden = np.dot(activations[l].T, delta) / X.shape[0]
                db_hidden = np.sum(delta, axis=0) / X.shape[0]

                # apply Adam updates to weights
                self.m_weights[l] = self.beta2 * self.m_weights[l] + (1 - self.beta2) * dW_hidden
                self.v_weights[l] = self.beta3 * self.v_weights[l] + (1 - self.beta3) * (dW_hidden ** 2)
                m_w_hat = self.m_weights[l] / (1 - self.beta2**time)
                v_w_hat = self.v_weights[l] / (1 - self.beta3**time)
                self.weights[l] -= (self.eta2 * m_w_hat) / (np.sqrt(v_w_hat) + self.epsilon2)

                # apply Adam updates to bias
                self.m_biases[l] = self.beta2 * self.m_biases[l] + (1 - self.beta2) * db_hidden
                self.v_biases[l] = self.beta3 * self.v_biases[l] + (1 - self.beta3) * (db_hidden ** 2)
                m_b_hat = self.m_biases[l] / (1 - self.beta2**time)
                v_b_hat = self.v_biases[l] / (1 - self.beta3**time)
                self.biases[l] -= (self.eta2 * m_b_hat) / (np.sqrt(v_b_hat) + self.epsilon2)

            self.weights_history.append([w.copy() for w in self.weights])
            self.biases_history.append([b.copy() for b in self.biases])
            
        return self
        
    def _get_shuffled(self, n_samples, X, y):
        permutation = np.random.permutation(n_samples)
        X_shuffled = X[permutation]
        y_shuffled = y[permutation]
        return X_shuffled, y_shuffled
    
    def _set_epoch_loss(self, y, activation):
        epoch_loss = self._compute_loss(y, activation)
        self.loss_history.append(epoch_loss)
        return self
    
    def _store_preds(self, X, y, activations, t, epoch):
        preds = {}
        activations, _ = self._forward_pass(X)
        self._set_epoch_loss(y, activations[-1])
        preds[f'activation {t}, {epoch}'] = activations[-1].flatten()
        self.estimator_preds.append(preds) # store the preds
        return self
    
    def _fit_MLP(self, t, X, residuals): #activations are multipled by learning rate and alpha
        X = np.asarray(X)
        y = np.asarray(residuals).reshape(-1, 1) #changing to y for easy coding
        n_samples, n_features = X.shape
        if self.batch_size is None: self.batch_size = n_samples #if no batch, set to number of samples
        
        # Initialize parameters
        self._initialize_parameters(n_features)
        
        self.weights_history.append([w.copy() for w in self.weights])
        self.biases_history.append([b.copy() for b in self.biases])
        activations, _ = self._forward_pass(X)
        initial_loss = self._compute_loss(y, activations[-1])
        self.loss_history.append(initial_loss)
        
        # global time step for Adam bias correction
        time = 0
        
        # Training loop
        if self.optimiser is None:
            for epoch in range(self.n_epochs):
            # Shuffle data: -------------------------------------------
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)
                # Mini-batch loop
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i:i + self.batch_size]
                    y_batch = y_shuffled[i:i + self.batch_size]
                    # Forward pass
                    activations, zs = self._forward_pass(X_batch) #output for model prediction
                    # Backward pass
                    self._backward_pass(X_batch, y_batch, activations, zs)
                    self._store_preds(X_batch, y_batch, activations, t, epoch)
                #end loop---------------------------------------------------
        
        if self.optimiser is 'SGD':
            for epoch in range(self.n_epochs):
            # shuffle datasets
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)

                # mini-batch loop
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i : i + self.batch_size]
                    y_batch = y_shuffled[i : i + self.batch_size]
                    #forward prop
                    activations, zs = self._forward_pass(X_batch)
                    #back prop
                    self._backward_pass(X_batch, y_batch, activations, zs)

                    #store results
                    self._store_preds(X_batch, y_batch, activations, t, epoch)

        if self.optimiser is 'adam':
             for epoch in range(self.n_epochs):
                # Shuffle data:  Gradient descent-----------------------------------
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)
                
                #mini batch loop
                for i in range(0, n_samples, self.batch_size):
                    time += 1
                    X_batch = X_shuffled[i : i + self.batch_size]
                    y_batch = y_shuffled[i : i + self.batch_size]

                    # 1. forward prop
                    activations, zs = self._forward_pass(X_batch)
                    # Backward prop
                    self._backward_pass(X_batch, y_batch, activations, zs, time)
                    #store results
                    self._store_preds(X_batch, y_batch, activations, t, epoch)

        return self
    
    #=======================================DECAY FUNCTIONS=======================================
    #'exponential', 'gradient'(simple Adagrad), 'beta', 
    def _exponential_decay(self, t): return self.initial_learning_rate * (np.exp(-self.decay * t))
    
    def _expSchedule_decay(self, t): return self.decay * (self.initial_learning_rate ** t)#exponential decay from 0.1 to 0.01 over 100 rounds

    def _gradient_decay(self, gradients): return self.learning_rate / (np.sum(gradients**2))    

    def _beta_decay(self, t): return self.beta_dist[t]
    
    def _logistic_decay(self, t): return self.initial_learning_rate/(1+ np.exp(-self.decay * (t -(self.initial_learning_rate/2)))) #use high values of decay for a less than 0 eta
    
    def _linear_decay(self, t): return self.initial_learning_rate - (self.decay * t) #use low values of decay else whacky results ensue
    
    def _polynomial_decay(self, t): return 1 - (t / self.n_estimators)**self.power
    
    def _inverse_time_decay(self, t): return self.initial_learning_rate / (1 + self.decay * math.floor(t/self.decay_steps))

    def _step_decay(self, t): return self.initial_learning_rate * self.decay**(math.floor((1+t)/self.decay_steps)) #higher decay, smaller schedule
    
    def _adaGrad_decay(self, gradients): #gradients are residuals
#         print('doing ADAGRAD')
        self.G += gradients**2
        self.eta = self.initial_learning_rate / (np.sqrt(self.G + self.epsilon3)) * gradients  #need new epsilon
        return self.eta
    
    def _eta_decay(self, t, residuals):
        if self.decay_function is 'adaGrad': self.eta = self._adaGrad_decay(residuals)
        elif t is 0: self.learning_rate = self.learning_rate #skip decay on round 1
        elif self.decay_function is 'exponential': self.learning_rate = self._exponential_decay(t)
        elif self.decay_function is 'exponential2': self.learning_rate = self._expSchedule_decay(t)
        elif self.decay_function is 'gradient': self.learning_rate = self._gradient_decay(residuals)
        elif self.decay_function is 'beta': self.learning_rate = self._beta_decay(t)
        elif self.decay_function is 'logistic': self.learning_rate = self._logistic_decay(t)
        elif self.decay_function is 'linear': self.learning_rate = self._linear_decay(t)
        elif self.decay_function is 'polynomial': self.learning_rate = self._polynomial_decay(t)
        elif self.decay_function is 'inverse': self.learning_rate = self._inverse_time_decay(t)
        elif self.decay_function is 'step': self.learning_rate = self._step_decay(t)
#         elif self.decay_function is 'adaGrad': self.eta = self._adaGrad_decay(residuals)
            
        if self.learning_rate is 0: self.learning_rate += 0.00001  #to avoid zeroing out the learning. 
        return self
    
    def _get_best_distribution(self, data):
        dist_names = ["norm", "exponweib", "weibull_max", "weibull_min", "genextreme", "beta", "gamma", 'expon', 'cauchy', 'laplace']
        dist_results = []
        params = {}

        for dist_name in dist_names:
            dist = getattr(st, dist_name)
            param = dist.fit(data)
            params[dist_name] = param

            # Apply the Kolmogorov-Smirnov test
            D, p = st.kstest(data, dist_name, args=param)
#             print(f"p value for {dist_name} = {p}")
            dist_results.append((dist_name, p))
#         print(params)
        # Select the best fitting distribution based on the highest p-value
        best_dist, best_p = max(dist_results, key=lambda item: item)
        print(f"Best fitting distribution: {best_dist}")
#         print(f"Best p value: {best_p}")
#         print(f"Parameters for the best fit: {params[best_dist]}")

        return best_dist, best_p, params[best_dist]

    def _set_distribution(self, best_dist, params):
        distribution = getattr(st, best_dist)
        dist = distribution(*params)
        linespace = np.linspace(0, 1, 100)
        self.distribution = dist.pdf(linespace)
        return self.distribution
    
    #=======================================Fit FUNCTIONS=======================================
    def _early_stopping(self, X, y, t):
        val_score = self.score(X, y)
        self.validation_scores.append(val_score)
#         print('Current score: ', val_score, '\nBest score: ', self.best_val_score, '\nTolerance: ', self.best_val_score - self.tolerance)
        if val_score < self.best_val_score - self.tolerance:
#             print('continue')
            self.best_val_score = val_score
            self.validation_scores.append(val_score)
            self.best_iteration = t + 1 #iterations are 0 inclusive, so need an extra 1
            self.no_improvement_count = 0 #reset the clock
        else: self.no_improvement_count += 1 #if going badly, lose patience
        
        self.n_estimators = self.best_iteration #update number of estimators so the prediction doesn't go awry

        if self.no_improvement_count >= self.patience:
            print(f'Early stopping at iteration {t+1}. Best validation {self.measure}: {self.best_val_score}')
            self.n_estimators = self.best_iteration #update number of estimators so the prediction doesn't go awry
            return True
            
        return False

    def _update_predictions(self, y_pred, feature_index, threshold, left_value, right_value, X):
        return np.array([y_pred[i] + (self.eta[i] * left_value) if X[:,feature_index][i] <= threshold else y_pred[i] + (self.eta[i] * right_value) for i in range(len(X[:,feature_index]))])  
    
    def fit(self, X, y):
#         print(self.eta)
        if self.validation_fraction is not None: #start early stopping
            X, X_val, y, y_val = modsel.train_test_split(X, y, test_size = self.validation_fraction, random_state = self.seed)
        
        self.initial_pred = self._initialize_predictions(y)
        self.current_pred = np.full(len(y), self.initial_pred)
        self.G = np.zeros(len(y))
                
        for t in range(self.n_estimators): #growing t trees... ba-dump, tiss
            if t is 0: self.eta = np.full(len(y), self.initial_learning_rate)
            if self.decay_function is not 'adaGrad':
                self.eta = np.full(len(y), self.learning_rate)
                
#             print('self.eta: ',self.eta)
            self.eta_history.append(self.eta) #storing for schedule analysis. it'll be a bit annoying to use later as array, but when fixed, can take any value
            #========================  
            residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
            #========================
            if self.classifier is 'stump': self._fit_stump(t, X, residuals)
            elif self.classifier is 'tree': self._fit_tree(t, X, residuals)
            elif self.classifier is 'MLP': self._fit_MLP(t, np.asarray(StandardScaler().fit_transform(X)), residuals)
                        
            #check early stopping criteria
            if self.validation_fraction is not None: 
#                 print(f'checking eraly stopping at round {t}')
                stop = self._early_stopping(X_val, y_val, t)
                if stop is True: break
            
            #now decay eta
            if self.decay_function is not None:
                self._eta_decay(t, residuals)
            
        #find best distributions for eta if using adagrad
        if self.decay_function is 'adaGrad':
            best_dist, best_p, params = self._get_best_distribution(self.eta_history[-1])
            self.dist = self._set_distribution(best_dist, params)
            #now, when predicting, consult the dist and get etas for each value
        
        return self
    
    #=======================================PREDICT FUNCTIONS=======================================
    #self.eta_testing_history =, self.testing_preds
    def _predict_sample(self, x, node): #goes to bottom of tree to get prediction, will need to do update_predictions
        if node['is_leaf']: return node['value']
        if x[node['feature']] <= node['threshold']:
            return self._predict_sample(x, node['left'])
        else: return self._predict_sample(x, node['right'])
    
    def _adaptive_update_preds(self, y_pred, feature_index, threshold, left_value, right_value, X):
        return np.array([y_pred[i] + (self.test_eta[i] * left_value) if X[:,feature_index][i] <= threshold else y_pred[i] + (self.test_eta[i] * right_value) for i in range(len(X[:,feature_index]))])  

    def _predict_train(self, X): #for adative decay, this avoids doubleling the descent on training
        y_pred = np.full(len(X), self.initial_pred)
        if self.classifier is 'stump':
            for stump in self.stumps:
                y_pred = self._update_predictions(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
        
        elif self.classifier is 'tree': 
            for t in range(len(self.trees)):
                predictions = [] #np.zeros(len(X))
                current_tree = self.trees[t][f'tree {t}']
                
                for x in X:
                    predictions.append(self._predict_sample(x, current_tree))
                
                predictions = np.array(predictions) * self.eta[0]  #self.learning_rate #will need to get indices and mask eta to work this out
                y_pred += predictions
                self.testing_preds.append(y_pred)
        
        elif self.classifier is 'MLP':
            X = StandardScaler().fit_transform(X)
            
            for t in range(self.n_estimators):
                X = np.asarray(X)
                activations, _ = self._forward_pass(X)
                predictions = activations[-1].flatten()
                predictions *= self.eta[0] #self.learning_rate #updated during forward and back prop
                y_pred += predictions #add to y_pred
                self.testing_preds.append(y_pred)
        return y_pred
    
    def _predict(self, predictions, y_pred, eta):
        predictions = np.array(predictions) * eta  #self.learning_rate #will need to get indices and mask eta to work this out
        y_pred += predictions
        self.testing_preds.append(y_pred)
        return y_pred

    def predict(self, X): #if decay is being used, I need to multiply each prediction by the relevant eta_history list
        y_pred = np.full(len(X), self.initial_pred)
        self.test_eta = np.full(len(X), self.initial_learning_rate)
        pdf = self.distribution
            
        if self.classifier is 'stump':
            for stump in self.stumps:
                if self.decay_function is 'adaGrad': #does adaptive decay on the stump
                    y_pred = self._adaptive_update_preds(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
                    self.testing_preds.append(y_pred)
                    self.test_eta = dist.pdf(y_pred)  #need to use a different eta to avoid doubling the gradient descent
                else: 
                    y_pred = self._update_predictions(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
                    self.testing_preds.append(y_pred)
          
        #iterate through each tree, get predictsion, weight them by learning rate and add the tree predictions to the total prediction (weight by gamme later)
        elif self.classifier is 'tree': 
            for t in range(len(self.trees)):
                predictions = [] #np.zeros(len(X))
                current_tree = self.trees[t][f'tree {t}']
                
                for x in X:
                    predictions.append(self._predict_sample(x, current_tree))
                
                if self.decay_function is 'adaGrad': 
                    y_pred =self._predict(predictions, y_pred, self.test_eta)
#                     predictions = np.array(predictions) * self.test_eta  #self.learning_rate #will need to get indices and mask eta to work this out
#                     y_pred += predictions
#                     self.testing_preds.append(y_pred)
                    self.test_eta = dist.pdf(y_pred)
                    
                else: y_pred = self._predict(predictions, y_pred, self.learning_rate)
#                     predictions = np.array(predictions) * self.learning_rate  #self.learning_rate #will need to get indices and mask eta to work this out
#                     y_pred += predictions
#                     self.testing_preds.append(y_pred)
        
        elif self.classifier is 'MLP':
            X = StandardScaler().fit_transform(X)
            
            for t in range(self.n_estimators):
                X = np.asarray(X)
                activations, _ = self._forward_pass(X)
                predictions = activations[-1].flatten()
                
                if self.decay_function is 'adaGrad':
                    y_pred =self._predict(predictions, y_pred, self.test_eta)
#                     predictions *= self.test_eta #self.learning_rate #updated during forward and back prop
#                     y_pred += predictions #add to y_pred
#                     self.testing_preds.append(y_pred)
                    self.test_eta = dist.pdf(y_pred)
                
                else: y_pred = self._predict(predictions, y_pred, self.learning_rate)
#                     predictions *= self.learning_rate #self.learning_rate #updated during forward and back prop
#                     y_pred += predictions #add to y_pred
#                     self.testing_preds.append(y_pred)
          
        return y_pred   
        
    def score(self, X, y): #for grid search and CV etc
        y_pred = self.predict(X)
        if self.measure is 'MSE': return np.mean((y - y_pred)**2)
        elif self.measure is 'RMSE': return np.sqrt(np.mean((y - y_pred) ** 2))
        elif self.measure is 'MAE': return np.mean(np.abs(y - y_pred))
        elif self.measure is 'MAPE': return np.mean(np.abs((y - y_pred) / y))
        elif self.measure is 'R2': return 1 - (np.sum((y - y_pred) ** 2) / np.sum((y - np.mean(y)) ** 2))
        return 